In [ ]:
# Practical 9: Slot Filling using Recurrent Neural Networks
#Code by Parthiv Abhani

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split

# ---------------------------------------------------------
# 1. Sample Spoken Language Understanding Dataset
# ---------------------------------------------------------

sentences = [
    "book a flight from delhi to mumbai",        # 7 words
    "book a flight from mumbai to delhi",        # 7 words
    "i want a flight from delhi to bangalore",   # 8 words
    "find flights from pune to delhi",           # 5 words
    "show flights from mumbai to chennai",       # 5 words
    "book a ticket from bangalore to hyderabad", # 7 words
    "i need a flight from chennai to pune",      # 8 words
    "find a flight from delhi to goa",           # 7 words
    "book flight from goa to mumbai",            # 6 words
    "i want to travel from pune to bangalore",   # 8 words
    "find flights from hyderabad to delhi",      # 6 words
    "book a flight from chennai to mumbai",      # 7 words
    "show me flights from delhi to pune",        # 7 words
    "i need a ticket from mumbai to goa",        # 8 words
    "book a flight from bangalore to delhi",     # 7 words
    "find flights from goa to chennai",          # 6 words
    "i want a flight from hyderabad to mumbai",  # 8 words
    "book ticket from pune to chennai",          # 5 words
    "find a flight from delhi to hyderabad",     # 7 words
    "show flights from mumbai to bangalore"      # 6 words
]

# Slot labels perfectly aligned to the word count of each sentence
labels = [
    ["O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "O", "B-FROM", "O", "B-TO"],
    ["O", "O", "O", "B-FROM", "O", "B-TO"]
]

# ---------------------------------------------------------
# 2. Tokenize Sentences
# ---------------------------------------------------------

tokenizer = Tokenizer(lower=True, filters='')

tokenizer.fit_on_texts(sentences)

X = tokenizer.texts_to_sequences(sentences)

MAX_LEN = max(len(x) for x in X)

X = pad_sequences(
    X,
    maxlen=MAX_LEN,
    padding='post'
)

# ---------------------------------------------------------
# 3. Encode Slot Labels
# ---------------------------------------------------------

label_names = ["O", "B-FROM", "B-TO"]

label_to_id = {
    label: i for i, label in enumerate(label_names)
}

y = []

for sentence_labels in labels:
    encoded = [label_to_id[label] for label in sentence_labels]

    while len(encoded) < MAX_LEN:
        encoded.append(label_to_id["O"])

    y.append(encoded)

y = np.array(y)

print("Vocabulary Size:", len(tokenizer.word_index) + 1)
print("Maximum Sentence Length:", MAX_LEN)

# ---------------------------------------------------------
# 4. Split Dataset
# ---------------------------------------------------------

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# ---------------------------------------------------------
# 5. Build LSTM Sequence Labeling Model
# ---------------------------------------------------------

model = tf.keras.Sequential([
    tf.keras.layers.Embedding(
        input_dim=len(tokenizer.word_index) + 1,
        output_dim=32,
        input_length=MAX_LEN
    ),

    tf.keras.layers.LSTM(
        64,
        return_sequences=True
    ),

    tf.keras.layers.Dense(
        32,
        activation='relu'
    ),

    tf.keras.layers.Dense(
        len(label_names),
        activation='softmax'
    )
])

# ---------------------------------------------------------
# 6. Compile Model
# ---------------------------------------------------------

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

# ---------------------------------------------------------
# 7. Train Model
# ---------------------------------------------------------

history = model.fit(
    X_train,
    y_train,
    epochs=30,
    batch_size=4,
    validation_split=0.2,
    verbose=1
)

# ---------------------------------------------------------
# 8. Evaluate Model
# ---------------------------------------------------------

loss, accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\nTest Accuracy:", round(accuracy * 100, 2), "%")
print("Test Loss:", round(loss, 4))

# ---------------------------------------------------------
# 9. Test Slot Filling on New Sentence
# ---------------------------------------------------------

test_sentence = "book a flight from delhi to mumbai"

test_sequence = tokenizer.texts_to_sequences([test_sentence])

test_sequence = pad_sequences(
    test_sequence,
    maxlen=MAX_LEN,
    padding='post'
)

prediction = model.predict(
    test_sequence,
    verbose=0
)

predicted_labels = np.argmax(prediction[0], axis=1)

words = test_sentence.split()

print("\nSlot Filling Result:")
print("-" * 35)

for i, word in enumerate(words):
    print(
        f"{word:12s} -> {label_names[predicted_labels[i]]}"
    )